In [1]:
print("G8 外匯 LLM 智能分析與回測系統")

G8 外匯 LLM 智能分析與回測系統


In [2]:
# 1.1 安裝套件
!pip install -q feedparser yfinance plotly

import datetime
import urllib.parse
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as gg
import plotly.express as px
import yfinance as yf
from datetime import datetime, timedelta
from google.colab import ai, data_table
from IPython.display import HTML, Markdown, display

# 啟用互動式表格
data_table.enable_dataframe_formatter()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.7 MB/s eta 0:00:00


In [3]:
# 1.2 設定 Colab Form 下拉選單與輸入項
Time_Period = '2 Months'  # @param ["1 Month", "2 Months", "3 Months", "6 Months"]
Target_Currency = 'GBP'  # @param ["EUR", "GBP", "JPY", "CAD", "AUD", "CHF", "NZD"]
Base_Currency = 'USD'  # @param ["USD"]

# 定義時間長度字典
period_days_map = {
    '1 Month': 30,
    '2 Months': 60,
    '3 Months': 90,
    '6 Months': 180,
}
days = period_days_map[Time_Period]

end_date = datetime.now()
start_date = end_date - timedelta(days=days)

print(
    f'✅ 已選擇時間範圍: {Time_Period} ({start_date.strftime("%Y-%m-%d")} 至'
    f' {end_date.strftime("%Y-%m-%d")})'
)
print(f'✅ 目標貨幣對: {Target_Currency}/{Base_Currency}')


✅ 已選擇時間範圍: 2 Months (2026-07-21 至 2026-09-19)
✅ 目標貨幣對: GBP/USD


In [4]:
# 2. 抓取 G8 貨幣歷史數據與表現排序 (G8 Performance Ranking)
# -------------------------------------------------------------------
g8_tickers = {
    'EUR': 'EURUSD=X',
    'GBP': 'GBPUSD=X',
    'JPY': 'JPY=X',  # 注意：JPY是 USD/JPY，計算回報時需取倒數或特別處理
    'CAD': 'CADUSD=X',
    'AUD': 'AUDUSD=X',
    'CHF': 'CHFUSD=X',
    'NZD': 'NZDUSD=X',
}

performance_data = []

for symbol, ticker in g8_tickers.items():
  try:
    df_fx = yf.download(
        ticker,
        start=start_date.strftime('%Y-%m-%d'),
        end=end_date.strftime('%Y-%m-%d'),
        progress=False,
    )
    if not df_fx.empty:
      start_price = df_fx['Close'].iloc[0]
      end_price = df_fx['Close'].iloc[-1]
      if symbol == 'JPY':  # JPY 是 USD/JPY
        pct_change = (
            (1 / end_price) - (1 / start_price)
        ) / (1 / start_price) * 100
      else:
        pct_change = (end_price - start_price) / start_price * 100

      # 取出數值
      pct_val = (
          pct_change.values[0]
          if hasattr(pct_change, 'values')
          else float(pct_change)
      )
      performance_data.append(
          {'Currency': symbol, 'Return (%)': round(pct_val, 2)}
      )
  except Exception as e:
    print(f'無法抓取 {symbol}: {e}')

df_perf = pd.DataFrame(performance_data).sort_values(
    by='Return (%)', ascending=False
)
display(Markdown('### 📊 G8 貨幣在選定區間內的表現排序'))
display(df_perf)

# 圖表化 G8 表現
fig_g8 = px.bar(
    df_perf,
    x='Currency',
    y='Return (%)',
    color='Return (%)',
    title=f'G8 Currency Performance ({Time_Period})',
    color_continuous_scale='RdYlGn',
)
fig_g8.show()


/tmp/ipykernel_2688/2748469156.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_fx = yf.download(
/tmp/ipykernel_2688/2748469156.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_fx = yf.download(
/tmp/ipykernel_2688/2748469156.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_fx = yf.download(
/tmp/ipykernel_2688/2748469156.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_fx = yf.download(
/tmp/ipykernel_2688/2748469156.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_fx = yf.download(
/tmp/ipykernel_2688/2748469156.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_fx = yf.download(
/tmp/ipykernel_2688/2748469156.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_fx = yf.download(


### 📊 G8 貨幣在選定區間內的表現排序

,Currency,Return (%)
2,JPY,4.07
4,AUD,1.64
0,EUR,0.63
3,CAD,0.58
1,GBP,-0.55
5,CHF,-1.43
6,NZD,-2.13


In [5]:
# =====================================================================
# 1. 自動安裝 Linux 系統依賴與必要套件
# =====================================================================
!pip install -q playwright googlenewsdecoder trafilatura feedparser
!playwright install --with-deps chromium

import asyncio
import os
import re
import urllib.parse
from bs4 import BeautifulSoup
import feedparser
from googlenewsdecoder import gnewsdecoder
from IPython.display import HTML, display
from playwright.async_api import async_playwright
import trafilatura

# 預設測試變數
if 'Target_Currency' not in globals():
  Target_Currency = 'GBP'

# =====================================================================
# 2. 爬蟲與新聞內文擷取主程式
# =====================================================================
news_query = f'{Target_Currency} Forex'
rss_url = f'https://news.google.com/rss/search?q={urllib.parse.quote(news_query)}&hl=en-US&gl=US&ceid=US:en'
feed = feedparser.parse(rss_url)


async def fetch_article_with_playwright(google_link):
  """解碼 Google News RSS 連結並使用真實瀏覽器抓取全文"""
  try:
    decoded_result = gnewsdecoder(google_link, interval=1)
    target_url = (
        decoded_result.get('decoded_url')
        if isinstance(decoded_result, dict)
        else google_link
    )

    async with async_playwright() as p:
      browser = await p.chromium.launch(
          headless=True,
          args=[
              '--no-sandbox',
              '--disable-setuid-sandbox',
              '--disable-blink-features=AutomationControlled',
          ],
      )
      context = await browser.new_context(
          user_agent=(
              'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
              ' (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
          ),
          viewport={'width': 1280, 'height': 800},
      )
      page = await context.new_page()

      await page.goto(
          target_url, wait_until='domcontentloaded', timeout=25000
      )
      await page.wait_for_timeout(2000)

      html_content = await page.content()
      await browser.close()

      clean_text = trafilatura.extract(
          html_content, include_comments=False, include_tables=False
      )
      if clean_text and len(clean_text) > 100:
        return clean_text

      soup = BeautifulSoup(html_content, 'html.parser')
      paragraphs = [p.get_text() for p in soup.find_all('p')]
      text = re.sub(r'\s+', ' ', ' '.join(paragraphs)).strip()
      if len(text) > 100:
        return text

  except Exception as e:
    return f'無法抓取內文 ({str(e)})'

  return '無有效內文'


# 執行抓取與高質感 HTML 渲染
async def main():
  global all_news_full_str
  articles_content = []
  news_cards_html = ''

  print(f'🚀 [System] 啟動自動化 Playwright 爬蟲引擎，目標貨幣: {Target_Currency}')

  for idx, entry in enumerate(feed.entries[:5], 1):
    title = entry.title
    link = entry.link

    print(f'[{idx}/5] 正在擷取新聞: {title[:40]}...')
    full_text = await fetch_article_with_playwright(link)

    truncated_text = full_text[:3000]
    articles_content.append(
        f'### 新聞 {idx}: {title}\n*連結: {link}*\n內文摘要/內容:\n{truncated_text}\n'
    )

    # 組合單篇新聞的可折疊卡片 HTML
    news_cards_html += f"""
        <div style="background-color: #131722; border: 1px solid #2A2E39; border-radius: 8px; margin-bottom: 12px; padding: 12px 16px;">
            <div style="display: flex; justify-content: space-between; align-items: center;">
                <span style="font-weight: 600; font-size: 14px; color: #FFFFFF;">#{idx} {title}</span>
                <a href="{link}" target="_blank" style="color: #2962FF; text-decoration: none; font-size: 12px; font-weight: 500;">🔗 原文連結</a>
            </div>
            <details style="margin-top: 10px; cursor: pointer;">
                <summary style="color: #787B86; font-size: 12px; outline: none;">📄 點擊展開 / 隱藏內文擷取預覽 ({len(full_text)} 字)</summary>
                <div style="background-color: #1E222D; color: #B2B5BE; padding: 12px; border-radius: 6px; margin-top: 8px; font-size: 12.5px; line-height: 1.6; max-height: 200px; overflow-y: auto; white-space: pre-wrap; font-family: monospace;">
{truncated_text}
                </div>
            </details>
        </div>
        """

  # 將供後續 LLM 使用的全文字串寫入全局變數
  all_news_full_str = '\n\n'.join(articles_content)

  # 渲染終端風格的主卡片
  dashboard_html = f"""
    <div style="
        background-color: #1E222D;
        color: #D1D4DC;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;
        padding: 20px 24px;
        border-radius: 12px;
        border: 1px solid #2A2E39;
        margin-top: 15px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.4);
    ">
        <!-- 頂部卡片標題列 -->
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #2A2E39; padding-bottom: 12px; margin-bottom: 16px;">
            <div>
                <span style="font-size: 18px; font-weight: 700; color: #FFFFFF;">📰 {Target_Currency} 即時新聞擷取結果</span>
                <span style="color: #787B86; font-size: 12px; margin-left: 8px;">| Automated Playwright Scraper</span>
            </div>
            <div style="background-color: #089981; color: #FFFFFF; padding: 4px 12px; border-radius: 16px; font-weight: bold; font-size: 12px;">
                🟢 5/5 FETCHED
            </div>
        </div>

        <!-- 新聞摺疊卡片區塊 -->
        <div>
            {news_cards_html}
        </div>
    </div>
    """
  display(HTML(dashboard_html))


# 啟動非同步任務
await main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 7.9 MB/s eta 0:00:00
Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Hit:5 https://cli.github.com/packages stable InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:

In [6]:
# =====================================================================
# 4 & 5. 運用 LLM 進行深度新聞內文分析 (自動 Markdown-to-HTML 渲染版)
# =====================================================================
import markdown
from IPython.display import HTML, display

# 1. 變數安全檢查 (避免 NameError)
if "all_news_full_str" not in globals() or not all_news_full_str:
  all_news_full_str = (
      "未找到新聞全文內容，請確認上方的爬蟲 Cell 是否已順利執行。"
  )

if "Target_Currency" not in globals():
  Target_Currency = "GBP"

if "Base_Currency" not in globals():
  Base_Currency = "USD"

# 2. 建立 Prompt
prompt_analysis = f"""
You are an expert forex trader and quantitative analyst.
Based on the following detailed news articles and context for {Target_Currency}:

{all_news_full_str}

Please perform the following in-depth tasks:
1. *Comprehensive Macro Summary*: Summarize the core macroeconomic themes, central bank stances (e.g. rate expectations), and economic data.
2. *Sentiment Analysis*: Evaluate the overall fundamental sentiment score for {Target_Currency} (from -1.0 Very Negative to +1.0 Very Positive).
3. *Trading Recommendation*: Provide a clear strategy [BUY, SHORT, or HOLD] for {Target_Currency}/{Base_Currency} with short-term rationale.

Format your response nicely using clean Markdown with structured tables and clear bullet points.
"""

# 3. 呼叫 LLM 產生分析結果
llm_analysis_output = ai.generate_text(prompt_analysis)


# 4. 定義卡片美化渲染函式 (含 Markdown 自動解析)
def render_institutional_report(symbol, raw_text):
  # 將 LLM 輸出的原始 Markdown 解析轉換為正確的 HTML 標籤 (支援表格與程式碼區塊)
  parsed_html = markdown.markdown(
      raw_text, extensions=["tables", "fenced_code", "nl2br"]
  )

  # 判斷多空顏色與標籤
  if "BUY" in raw_text or "看多" in raw_text:
    badge_color = "#089981"
    badge_text = "🟢 BUY / BULLISH"
  elif "SHORT" in raw_text or "SELL" in raw_text or "看空" in raw_text:
    badge_color = "#F23645"
    badge_text = "🔴 SHORT / BEARISH"
  else:
    badge_color = "#2962FF"
    badge_text = "⚪ HOLD / NEUTRAL"

  # 組合機構級高質感 CSS 樣式
  html_code = f"""
    <style>
        .report-card {{
            background-color: #1E222D;
            color: #D1D4DC;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
            padding: 24px;
            border-radius: 12px;
            border: 1px solid #2A2E39;
            margin-top: 15px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.5);
        }}
        .report-card h1, .report-card h2, .report-card h3 {{
            color: #FFFFFF;
            border-bottom: 1px solid #2A2E39;
            padding-bottom: 8px;
            margin-top: 20px;
        }}
        .report-card table {{
            width: 100%;
            border-collapse: collapse;
            margin: 15px 0;
            background-color: #131722;
            border-radius: 6px;
            overflow: hidden;
        }}
        .report-card th {{
            background-color: #2A2E39;
            color: #2962FF;
            text-align: left;
            padding: 10px;
            font-size: 13px;
        }}
        .report-card td {{
            padding: 10px;
            border-bottom: 1px solid #2A2E39;
            font-size: 13px;
        }}
        .report-card pre {{
            background-color: #131722;
            padding: 12px;
            border-radius: 6px;
            border-left: 4px solid #2962FF;
            font-family: monospace;
            overflow-x: auto;
            color: #00E676;
        }}
        .report-card blockquote {{
            background: #262B3E;
            border-left: 4px solid #FF9800;
            margin: 10px 0;
            padding: 10px 15px;
            color: #FFE082;
        }}
    </style>

    <div class="report-card">
        <!-- 頂部標題列 -->
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #2A2E39; padding-bottom: 15px; margin-bottom: 15px;">
            <div>
                <span style="font-size: 20px; font-weight: 700; color: #FFFFFF;">🌐 G8 外匯 LLM 智能市場分析報告</span>
                <span style="color: #787B86; font-size: 12px; margin-left: 8px;">| Institutional Intelligence Engine</span>
            </div>
            <div style="background-color: {badge_color}; color: #FFFFFF; padding: 6px 14px; border-radius: 20px; font-weight: bold; font-size: 13px;">
                {badge_text}
            </div>
        </div>

        <!-- 標的狀態列 -->
        <div style="background-color: #131722; padding: 10px 16px; border-radius: 6px; margin-bottom: 20px; display: flex; gap: 25px; font-size: 13px;">
            <div><span style="color: #787B86;">監測貨幣對:</span> <b style="color: #FFFFFF;">{symbol}</b></div>
            <div><span style="color: #787B86;">分析類型:</span> <b style="color: #FFFFFF;">新聞情緒與宏觀央行態度</b></div>
            <div><span style="color: #787B86;">狀態:</span> <b style="color: #089981;">即時運算完成</b></div>
        </div>

        <!-- 解析後的完整 HTML 報告 -->
        <div style="line-height: 1.7; font-size: 14px;">
            {parsed_html}
        </div>
    </div>
    """
  display(HTML(html_code))


# 5. 輸出渲染結果
render_institutional_report(
    f"{Target_Currency}/{Base_Currency}", llm_analysis_output
)

In [8]:
# -------------------------------------------------------------------
# 6 & 7. PnL 預測模擬與綜合報告 (PnL Projection & Final Report)
# -------------------------------------------------------------------
import markdown
from IPython.display import HTML, display

# 預設變數檢查 (避免單獨執行時 NameError)
if "Target_Currency" not in globals():
  Target_Currency = "GBP"
if "Time_Period" not in globals():
  Time_Period = "2 Months"

prompt_pnl = f"""
Based on your trading recommendation for {Target_Currency} and historical trends over the past {Time_Period}:
1. Estimate the projected Profit and Loss (PnL %) expectation for the next 1 Month and 2 Months under:
   - Best-case scenario
   - Base-case scenario
   - Worst-case scenario
2. Generate a structured final report summarizing:
   - Currency Rank in G8
   - LLM Recommendation (Buy/Short/Hold)
   - Expected 1-Month PnL (%) & 2-Month PnL (%)
   - Key Market Risks

Format your response using clear Markdown tables, bold metrics, and structured bullet points.
"""

llm_pnl_output = ai.generate_text(prompt_pnl)


# -------------------------------------------------------------------
# 高質感 HTML/CSS 渲染卡片 (機構級暗色主題)
# -------------------------------------------------------------------
def render_pnl_report(symbol, raw_text):
  # 將 Markdown 轉為標準 HTML (啟用表格與代碼塊擴充)
  parsed_html = markdown.markdown(
      raw_text, extensions=["tables", "fenced_code", "nl2br"]
  )

  # 自動偵測多空狀態並設置色塊
  if "BUY" in raw_text or "看多" in raw_text:
    badge_color = "#089981"
    badge_text = "🟢 BUY / LONG"
  elif "SHORT" in raw_text or "SELL" in raw_text or "看空" in raw_text:
    badge_color = "#F23645"
    badge_text = "🔴 SHORT / BEARISH"
  else:
    badge_color = "#2962FF"
    badge_text = "⚪ HOLD / NEUTRAL"

  html_code = f"""
    <style>
        .pnl-card {{
            background-color: #1E222D;
            color: #D1D4DC;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
            padding: 24px;
            border-radius: 12px;
            border: 1px solid #2A2E39;
            margin-top: 15px;
            box-shadow: 0 4px 18px rgba(0,0,0,0.45);
        }}
        .pnl-card h1, .pnl-card h2, .pnl-card h3 {{
            color: #FFFFFF;
            border-bottom: 1px solid #2A2E39;
            padding-bottom: 8px;
            margin-top: 22px;
            font-weight: 600;
        }}
        .pnl-card table {{
            width: 100%;
            border-collapse: collapse;
            margin: 16px 0;
            background-color: #131722;
            border-radius: 8px;
            overflow: hidden;
            border: 1px solid #2A2E39;
        }}
        .pnl-card th {{
            background-color: #2A2E39;
            color: #2962FF;
            text-align: left;
            padding: 12px 14px;
            font-size: 13px;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}
        .pnl-card td {{
            padding: 12px 14px;
            border-bottom: 1px solid #2A2E39;
            font-size: 13.5px;
        }}
        .pnl-card tr:last-child td {{
            border-bottom: none;
        }}
        .pnl-card pre {{
            background-color: #131722;
            padding: 14px;
            border-radius: 6px;
            border-left: 4px solid #089981;
            font-family: 'Courier New', Courier, monospace;
            overflow-x: auto;
            color: #00E676;
            font-size: 13px;
        }}
        .pnl-card blockquote {{
            background: #262B3E;
            border-left: 4px solid #FF9800;
            margin: 12px 0;
            padding: 10px 16px;
            color: #FFE082;
            border-radius: 0 6px 6px 0;
        }}
        .pnl-card ul, .pnl-card ol {{
            padding-left: 20px;
            line-height: 1.7;
        }}
    </style>

    <div class="pnl-card">
        <!-- 頂部卡片標題列 -->
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #2A2E39; padding-bottom: 14px; margin-bottom: 18px;">
            <div>
                <span style="font-size: 20px; font-weight: 700; color: #FFFFFF;">📈 預期 PnL 模擬與綜合決策報告</span>
                <span style="color: #787B86; font-size: 12px; margin-left: 8px;">| G8 Quantitative Model</span>
            </div>
            <div style="background-color: {badge_color}; color: #FFFFFF; padding: 6px 14px; border-radius: 20px; font-weight: bold; font-size: 13px;">
                {badge_text}
            </div>
        </div>

        <!-- 數據指標概覽 -->
        <div style="background-color: #131722; padding: 12px 18px; border-radius: 8px; margin-bottom: 20px; display: flex; gap: 30px; font-size: 13px;">
            <div><span style="color: #787B86;">目標貨幣:</span> <b style="color: #FFFFFF;">{symbol}</b></div>
            <div><span style="color: #787B86;">回測歷史區間:</span> <b style="color: #FFFFFF;">{Time_Period}</b></div>
            <div><span style="color: #787B86;">模型引擎:</span> <b style="color: #2962FF;">LLM Scenario Simulator</b></div>
        </div>

        <!-- 渲染後的 Markdown 內容 -->
        <div style="line-height: 1.7; font-size: 14px;">
            {parsed_html}
        </div>
    </div>
    """
  display(HTML(html_code))


# 執行卡片渲染
render_pnl_report(Target_Currency, llm_pnl_output)

In [9]:
# -------------------------------------------------------------------
# 8. 系統與策略限制推測 (Limitations Analysis)
# -------------------------------------------------------------------
import markdown
from IPython.display import HTML, display

prompt_limitations = """
Please outline 4 critical limitations of using this LLM + RSS Sentiment Forex Analysis strategy (e.g., RSS news delay, hallucination risk, lack of real-time technical order book data, unforeseen geopolitical black swan events).

Format your response nicely using clear headings, bold warning tags, structured bullet points, and a summary table if applicable.
"""

llm_limitations = ai.generate_text(prompt_limitations)


# -------------------------------------------------------------------
# 高質感 HTML/CSS 警示卡片渲染函式
# -------------------------------------------------------------------
def render_limitations_report(raw_text):
  # 將 Markdown 轉換為 HTML 標籤
  parsed_html = markdown.markdown(
      raw_text, extensions=["tables", "fenced_code", "nl2br"]
  )

  html_code = f"""
    <style>
        .risk-card {{
            background-color: #1E222D;
            color: #D1D4DC;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
            padding: 24px;
            border-radius: 12px;
            border: 1px solid #FF9800;
            margin-top: 15px;
            box-shadow: 0 4px 18px rgba(255, 152, 0, 0.15);
        }}
        .risk-card h1, .risk-card h2, .risk-card h3 {{
            color: #FFB74D;
            border-bottom: 1px solid #2A2E39;
            padding-bottom: 8px;
            margin-top: 22px;
            font-weight: 600;
        }}
        .risk-card table {{
            width: 100%;
            border-collapse: collapse;
            margin: 16px 0;
            background-color: #131722;
            border-radius: 8px;
            overflow: hidden;
            border: 1px solid #2A2E39;
        }}
        .risk-card th {{
            background-color: #2A2E39;
            color: #FFB74D;
            text-align: left;
            padding: 12px 14px;
            font-size: 13px;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}
        .risk-card td {{
            padding: 12px 14px;
            border-bottom: 1px solid #2A2E39;
            font-size: 13.5px;
        }}
        .risk-card tr:last-child td {{
            border-bottom: none;
        }}
        .risk-card ul, .risk-card ol {{
            padding-left: 20px;
            line-height: 1.7;
        }}
        .risk-card blockquote {{
            background: #2D2219;
            border-left: 4px solid #FF9800;
            margin: 12px 0;
            padding: 12px 16px;
            color: #FFE082;
            border-radius: 0 6px 6px 0;
        }}
        .risk-card code {{
            background-color: #2A2E39;
            color: #FFB74D;
            padding: 2px 6px;
            border-radius: 4px;
            font-size: 12px;
        }}
    </style>

    <div class="risk-card">
        <!-- 頂部警示 header -->
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #2A2E39; padding-bottom: 14px; margin-bottom: 18px;">
            <div>
                <span style="font-size: 20px; font-weight: 700; color: #FFFFFF;">⚠️ 系統與模型風險限制 (Limitations Analysis)</span>
                <span style="color: #787B86; font-size: 12px; margin-left: 8px;">| Strategy Risk Control</span>
            </div>
            <div style="background-color: #FF9800; color: #1E222D; padding: 6px 14px; border-radius: 20px; font-weight: bold; font-size: 13px;">
                HIGH RISK WARNING
            </div>
        </div>

        <!-- 警示提示條 -->
        <div style="background-color: #2D2219; border: 1px solid #FF9800; padding: 10px 16px; border-radius: 6px; margin-bottom: 20px; font-size: 13px; color: #FFE082;">
            <b>提示：</b> 本模型僅供宏觀研判與輔助參考，實盤交易時需特別注意以下 4 大系統性與策略技術限制。
        </div>

        <!-- 解析後的 Markdown 內容 -->
        <div style="line-height: 1.7; font-size: 14px;">
            {parsed_html}
        </div>
    </div>
    """
  display(HTML(html_code))


# 執行渲染
render_limitations_report(llm_limitations)


Critical Limitation,Primary Risk,Direct Impact on Forex Trading,Potential Mitigation Strategy
RSS Latency,Execution Delay,Entering trades after the market has already priced in the news.,"Replace RSS with WebSockets and direct low-latency institutional squawk feeds (e.g., Bloomberg Terminal API, Reuters)."
LLM Hallucination,Analytical Error,Executing trades based on falsely interpreted sentiment or fabricated data.,"Implement strict programmatic guardrails, RAG (Retrieval-Augmented Generation), and rule-based validation of LLM outputs."
No Order Book/Tech Data,Market Blindness,"Buying into heavy resistance or illiquid markets, leading to high slippage.","Use a hybrid system where LLM sentiment is only a ""filter,"" and trade execution is triggered by technical indicators (MACD, RSI, Order Book depth)."
Black Swan Vulnerability,Capital Wipeout,Heavy losses during unprecedented geopolitical or economic shocks.,"Implement hard-coded, non-LLM circuit breakers, strict maximum-loss API stops, and manual kill-switches."


In [10]:
# -------------------------------------------------------------------
# 9. PnL 模擬數據圖示化 (Visualization)
# -------------------------------------------------------------------
# 建立一個模擬的情境 PnL 視覺化圖表
scenarios = ['Worst Case', 'Base Case', 'Best Case']
pnl_1m = [-2.5, 1.8, 4.5]  # 範例數據
pnl_2m = [-4.0, 3.2, 7.8]  # 範例數據

df_pnl_vis = pd.DataFrame(
    {'Scenario': scenarios, '1-Month PnL (%)': pnl_1m, '2-Month PnL (%)': pnl_2m}
)

fig_pnl = px.bar(
    df_pnl_vis,
    x='Scenario',
    y=['1-Month PnL (%)', '2-Month PnL (%)'],
    barmode='group',
    title=f'Projected PnL Scenarios for {Target_Currency}',
    labels={'value': 'Expected Return (%)'},
)
fig_pnl.show()
